# 📊 SENSEX Options Historical Data Pipeline
> **Source:** Groww API only — no synthetic data  
> **Output:** `sensex_weekly_data/sensex_options_YYYYMMDD.csv` (one file per weekly expiry)  
> **Merge:** `sensex_options_merged.csv` (combined dataset for backtest engine)

---

## 📋 Workflow
| Step | Description |
|------|-------------|
| **Cell 1** | Install / import dependencies |
| **Cell 2** | Configure API credentials & date range |
| **Cell 3** | Initialize Groww API & verify connection |
| **Cell 4** | Fetch SENSEX weekly expiry list |
| **Cell 5** | Fetch SENSEX index (cash) data |
| **Cell 6** | Fetch options data & save weekly CSV files |
| **Cell 7** | Merge all weekly files into one dataset |
| **Cell 8** | Verify & preview the merged dataset |

---
## Cell 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys

# Auto-install tqdm if missing
try:
    from tqdm.notebook import tqdm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm', '-q'])
    from tqdm.notebook import tqdm

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import os
import glob
from growwapi import GrowwAPI

print('✅ All dependencies loaded.')

---
## Cell 2 — Configuration
> ✏️ **Edit these values before running the pipeline.**

In [ ]:
# ── Groww API Credentials ────────────────────────────────────────────────────
USER_API_KEY = "eyJraWQiOiJaTUtjVXciLCJhbGciOiJFUzI1NiJ9.eyJleHAiOjI1NDg2NzA2OTQsImlhdCI6MTc2MDI3MDY5NCwibmJmIjoxNzYwMjcwNjk0LCJzdWIiOiJ7XCJ0b2tlblJlZklkXCI6XCIyYWM5YmY5MS1jYzVhLTQ4ZWQtYmFiZi1lYTU0MGYxNGM2YTlcIixcInZlbmRvckludGVncmF0aW9uS2V5XCI6XCJlMzFmZjIzYjA4NmI0MDZjODg3NGIyZjZkODQ5NTMxM1wiLFwidXNlckFjY291bnRJZFwiOlwiY2E4OGIyMzYtOGViNS00YzkxLTk5YzQtYmQ4OTgzN2ZlZjVjXCIsXCJkZXZpY2VJZFwiOlwiMTc3M2Q1MGEtNzU2ZC01NWIxLWEyZDQtYmU4YzFhMmEzYmZmXCIsXCJzZXNzaW9uSWRcIjpcImE4NDdlMDA3LWM3YWUtNDY5Ny1hMDMxLTNkZWNmZDUyNDhlYVwiLFwiYWRkaXRpb25hbERhdGFcIjpcIno1NC9NZzltdjE2WXdmb0gvS0EwYkU0a1gxVTh4cGdYS1F4dER0SnZQT1JSTkczdTlLa2pWZDNoWjU1ZStNZERhWXBOVi9UOUxIRmtQejFFQisybTdRPT1cIixcInJvbGVcIjpcImF1dGgtdG90cFwiLFwic291cmNlSXBBZGRyZXNzXCI6XCIyNDA5OjQwZjI6MjA4Zjo0NTgxOjNjZDg6NDU1NDphZWQ5OmYwNDUsMTYyLjE1OC41MS4xNzUsMzUuMjQxLjIzLjEyM1wiLFwidHdvRmFFeHBpcnlUc1wiOjI1NDg2NzA2OTQ5MzF9IiwiaXNzIjoiYXBleC1hdXRoLXByb2QtYXBwIn0.4dniz_YUUXpb3FArha_ElYjTQRWB6xQTnfKXk23hQK4dwzRTZlHpPjlvMMc-VxxYonLzQTwAsEaGYiEyTxsbUg"
USER_SECRET  = "U4isED)&@vy^Di9r&!hB7cw!65)z!yax"

# ── Date Range ───────────────────────────────────────────────────────────────
YEARS       = [2025]    # e.g. [2024, 2025]
START_MONTH = 1         # 1 = January
END_MONTH   = 2         # 2 = February

# ── Output Paths ─────────────────────────────────────────────────────────────
OUTPUT_DIR  = r"d:\Algotrade\AlgorithmicStockTrading\sensex_weekly_data"
MERGED_FILE = r"d:\Algotrade\AlgorithmicStockTrading\sensex_options_merged.csv"

# ── API Rate Limiting ─────────────────────────────────────────────────────────
REQUEST_DELAY  = 1     # seconds between requests
MAX_RETRIES    = 3
BACKOFF_FACTOR = 2

# ── Candle Interval ───────────────────────────────────────────────────────────
CANDLE_INTERVAL = "5minute"

print(f"📅 Fetching: {YEARS}  Months {START_MONTH}–{END_MONTH}")
print(f"📁 Output : {OUTPUT_DIR}")
print(f"📄 Merged : {MERGED_FILE}")

---
## Cell 3 — Initialize Groww API

In [ ]:
access_token = GrowwAPI.get_access_token(api_key=USER_API_KEY, secret=USER_SECRET)
groww = GrowwAPI(access_token)
print("✅ Groww API initialized successfully")

# Quick connection test
test = groww.get_expiries(
    exchange=groww.EXCHANGE_BSE,
    underlying_symbol="SENSEX",
    year=YEARS[0],
    month=START_MONTH
)
print(f"🔗 Connection verified — sample expiries: {test.get('expiries', [])[:3]}")

---
## Helper Functions

In [ ]:
def make_api_request_with_retry(func, *args, **kwargs):
    """API call with exponential backoff retry."""
    for attempt in range(MAX_RETRIES):
        try:
            response = func(*args, **kwargs)
            time.sleep(REQUEST_DELAY)
            return response
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait = REQUEST_DELAY * (BACKOFF_FACTOR ** attempt)
                print(f"  ⚠ Retry {attempt+1}/{MAX_RETRIES} in {wait}s — {str(e)[:60]}")
                time.sleep(wait)
            else:
                print(f"  ✗ Failed after {MAX_RETRIES} attempts: {str(e)[:80]}")
                raise

print("✅ Helper functions defined.")

---
## Cell 5 — Fetch SENSEX Weekly Expiry List

In [ ]:
all_expiries = []
month_list   = [(y, m) for y in YEARS for m in range(START_MONTH, END_MONTH + 1)]

for year, month in tqdm(month_list, desc='Fetching expiries'):
    try:
        resp = make_api_request_with_retry(
            groww.get_expiries,
            exchange=groww.EXCHANGE_BSE,
            underlying_symbol='SENSEX',
            year=year,
            month=month
        )
        if resp and 'expiries' in resp:
            all_expiries.extend(resp['expiries'])
            print(f"  ✓ {year}-{month:02d}: {resp['expiries']}")
        else:
            print(f"  ⚠ No expiries for {year}-{month:02d}")
    except Exception as e:
        print(f"  ✗ {year}-{month:02d}: {e}")

all_expiries = sorted(set(all_expiries))
print(f"\n📊 Total unique expiries: {len(all_expiries)}")
print(f"   {all_expiries}")

---
## Cell 6 — Fetch SENSEX Index (Cash) Data
> Used to populate the `index_close` column. Source: Groww API `EXCHANGE_BSE / SEGMENT_CASH / BSE-SENSEX`.

In [ ]:
earliest = datetime.strptime(all_expiries[0],  '%Y-%m-%d')
latest   = datetime.strptime(all_expiries[-1], '%Y-%m-%d')

index_start = (earliest - timedelta(days=10)).strftime('%Y-%m-%d')
index_end   = latest.strftime('%Y-%m-%d')
print(f"Fetching SENSEX index: {index_start} → {index_end}")

all_index_data = []
cs = datetime.strptime(index_start, '%Y-%m-%d')
fe = datetime.strptime(index_end,   '%Y-%m-%d')
chunks = []
while cs <= fe:
    ce = min(cs + timedelta(days=30), fe)
    chunks.append((cs, ce))
    cs = ce + timedelta(days=1)

for chunk_start, chunk_end in tqdm(chunks, desc='Index chunks'):
    try:
        resp = make_api_request_with_retry(
            groww.get_historical_candles,
            exchange=groww.EXCHANGE_BSE,
            segment=groww.SEGMENT_CASH,
            groww_symbol='BSE-SENSEX',
            start_time=chunk_start.strftime('%Y-%m-%d 09:15:00'),
            end_time=chunk_end.strftime('%Y-%m-%d 15:30:00'),
            candle_interval=groww.CANDLE_INTERVAL_DAY
        )
        if resp and 'candles' in resp:
            for c in resp['candles']:
                all_index_data.append({
                    'date_only': pd.to_datetime(c[0]).date(),
                    'sx_open':  c[1], 'sx_high': c[2],
                    'sx_low':   c[3], 'sx_close': c[4],
                })
    except Exception as e:
        print(f"  ✗ {chunk_start.date()}: {e}")

if all_index_data:
    index_df = pd.DataFrame(all_index_data)
    index_df['date_only'] = pd.to_datetime(index_df['date_only'])
    index_df = index_df.drop_duplicates('date_only').sort_values('date_only').reset_index(drop=True)
    print(f"\n✅ SENSEX index data: {len(index_df)} trading days")
    display(index_df.head())
else:
    index_df = pd.DataFrame()
    print("⚠ No index data — index_close will be NaN")

---
## Cell 7 — Fetch Options Data & Save Weekly CSVs
> ⏱️ This is the main long-running step. Each expiry fetches ~200-600 contracts × ~75 candles each.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

total_records    = 0
total_files      = 0
skipped_expiries = []
pipeline_start   = time.time()

for idx, expiry_date in enumerate(tqdm(all_expiries, desc='Processing expiries')):
    # ── Resume check: skip if file already exists with data ──────────────
    fname = f"sensex_options_{expiry_date.replace('-', '')}.csv"
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath) and os.path.getsize(fpath) > 1000:
        print(f"  ⏭ Skipping {fname} already exists ({os.path.getsize(fpath)/1024:.1f} KB)")
        total_files += 1
        continue

    print(f"\n{'─'*60}")
    print(f"  [{idx+1}/{len(all_expiries)}] Expiry: {expiry_date}")

    # ── Get contracts for this expiry ─────────────────────────────────────
    try:
        cr = make_api_request_with_retry(
            groww.get_contracts,
            exchange=groww.EXCHANGE_BSE,
            underlying_symbol='SENSEX',
            expiry_date=expiry_date
        )
    except Exception as e:
        print(f"  ✗ Contracts error: {e}")
        skipped_expiries.append(expiry_date)
        continue

    if not cr or 'contracts' not in cr:
        print(f"  ⚠ No contracts found")
        skipped_expiries.append(expiry_date)
        continue

    contracts = [c for c in cr['contracts'] if not c.endswith('-FUT')]
    print(f"  ✓ {len(contracts)} option contracts found")

    # ── Date range: 7 days before expiry → expiry day ────────────────────
    expiry_dt  = datetime.strptime(expiry_date, '%Y-%m-%d')
    start_time = (expiry_dt - timedelta(days=7)).strftime('%Y-%m-%d 09:15:00')
    end_time   = expiry_dt.strftime('%Y-%m-%d 15:30:00')

    # ── Fetch candles for each contract ───────────────────────────────────
    all_candle_data = []
    ok = skip = err = 0

    for contract in tqdm(contracts, desc=f'  Contracts {expiry_date}', leave=False):
        try:
            resp = make_api_request_with_retry(
                groww.get_historical_candles,
                exchange=groww.EXCHANGE_BSE,
                segment=groww.SEGMENT_FNO,
                groww_symbol=contract,
                start_time=start_time,
                end_time=end_time,
                candle_interval=CANDLE_INTERVAL
            )
            if resp and 'candles' in resp and len(resp['candles']) > 0:
                parts  = contract.split('-')
                strike = parts[-2] if len(parts) >= 4 else ''
                optype = parts[-1] if len(parts) >= 4 else ''
                expstr = parts[-3] if len(parts) >= 4 else ''
                for c in resp['candles']:
                    all_candle_data.append({
                        'timestamp':    c[0],
                        'open':         c[1], 'high': c[2],
                        'low':          c[3], 'close': c[4],
                        'volume':       c[5],
                        'oi':           c[6] if len(c) > 6 else np.nan,
                        'groww_symbol': contract,
                        'strike_price': strike,
                        'option_type':  optype,
                        'expiry':       expstr,
                        'expiry_date':  expiry_date,
                    })
                ok += 1
            else:
                skip += 1
        except Exception:
            err += 1

    print(f"  ✓ Fetched: {ok} contracts | skipped: {skip} | errors: {err}")
    print(f"  ✓ Total candles: {len(all_candle_data):,}")

    if not all_candle_data:
        skipped_expiries.append(expiry_date)
        continue

    # ── Build dataframe & enrich ──────────────────────────────────────────
    df = pd.DataFrame(all_candle_data)
    df['timestamp']   = pd.to_datetime(df['timestamp'])
    df['expiry_date'] = pd.to_datetime(df['expiry_date'])
    df = df.rename(columns={'groww_symbol': 'symbol'})

    df['date']       = df['timestamp']
    df['date_only']  = df['timestamp'].dt.normalize()
    df['time']       = df['timestamp'].dt.strftime('%H:%M')
    df['AM_PM']      = df['timestamp'].dt.strftime('%p')
    df['day']        = df['timestamp'].dt.strftime('%A')
    df['expiry_day'] = df['expiry_date'].dt.strftime('%A')
    df['month']      = df['timestamp'].dt.strftime('%B')
    df['month_num']  = df['timestamp'].dt.month

    df['dte_num'] = df.apply(
        lambda row: np.busday_count(row['timestamp'].date(), row['expiry_date'].date())
        if pd.notna(row['timestamp']) and pd.notna(row['expiry_date']) else np.nan,
        axis=1
    )
    df['DTE'] = df['dte_num'].apply(
        lambda x: 'ODTE' if x == 0 else f'{int(x)}DTE' if x <= 7 else '>7DTE'
    )

    # Merge real SENSEX index prices (NaN if date not in index data)
    if not index_df.empty:
        idf = index_df.copy()
        idf['date_only'] = pd.to_datetime(idf['date_only'])
        df = df.merge(idf[['date_only','sx_open','sx_high','sx_low','sx_close']],
                      on='date_only', how='left')
        df = df.rename(columns={'sx_close': 'index_close'})
    else:
        df['index_close'] = np.nan
        df['sx_open'] = df['sx_high'] = df['sx_low'] = np.nan

    df = df.sort_values(['timestamp','symbol']).reset_index(drop=True)

    # ── Save to CSV ───────────────────────────────────────────────────────
    df.to_csv(fpath, index=False)
    fsize = os.path.getsize(fpath) / (1024*1024)
    print(f"  💾 Saved: {fname}  ({len(df):,} rows, {fsize:.2f} MB)")
    total_files   += 1
    total_records += len(df)

elapsed = time.time() - pipeline_start
print(f"\n{'='*60}")
print(f"✅ Done! {total_files} files | {total_records:,} records | {str(timedelta(seconds=int(elapsed)))} elapsed")
if skipped_expiries:
    print(f"⚠ Skipped (no data): {skipped_expiries}")

---
## Cell 8 — Merge All Weekly Files

In [ ]:
csv_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'sensex_options_*.csv')))
print(f"📁 Found {len(csv_files)} weekly CSV files\n")

all_dfs       = []
total_records = 0

for fpath in tqdm(csv_files, desc='Reading weekly CSVs'):
    fname = os.path.basename(fpath)
    fsize = os.path.getsize(fpath) / (1024*1024)
    try:
        df   = pd.read_csv(fpath)
        total_records += len(df)
        all_dfs.append(df)
        print(f"  ✓ {fname:45s}  {len(df):>8,} rows  ({fsize:.2f} MB)")
    except Exception as e:
        print(f"  ✗ {fname}: {e}")

if not all_dfs:
    print("✗ No data to merge.")
else:
    print(f"\n🔧 Merging {len(all_dfs)} files ({total_records:,} rows)...")
    merged_df = pd.concat(all_dfs, ignore_index=True)

    for col in ['timestamp','expiry_date','date']:
        if col in merged_df.columns:
            merged_df[col] = pd.to_datetime(merged_df[col])

    sort_cols = [c for c in ['timestamp','symbol'] if c in merged_df.columns]
    if sort_cols:
        merged_df = merged_df.sort_values(sort_cols).reset_index(drop=True)

    merged_df.to_csv(MERGED_FILE, index=False)
    fsize = os.path.getsize(MERGED_FILE) / (1024*1024)
    print(f"\n💾 Saved merged file: {MERGED_FILE}  ({fsize:.2f} MB)")
    print(f"✅ Merged: {len(merged_df):,} total rows")

---
## Cell 9 — Verify & Preview Merged Dataset

In [ ]:
print("=" * 70)
print("  MERGED DATASET SUMMARY")
print("=" * 70)
print(f"Shape              : {merged_df.shape}")
print(f"Columns            : {list(merged_df.columns)}")
print(f"Date range         : {merged_df['timestamp'].min()} → {merged_df['timestamp'].max()}")
print(f"Unique symbols     : {merged_df['symbol'].nunique():,}")
print(f"Unique expiry dates: {merged_df['expiry_date'].nunique()}")
print(f"NaN in index_close : {merged_df['index_close'].isna().sum():,} ")
print(f"DTE distribution:")
print(merged_df['DTE'].value_counts().to_string())

print("\n--- First 5 rows ---")
display(merged_df[['timestamp','symbol','option_type','strike_price','open','high','low','close',
                   'volume','oi','DTE','index_close','expiry_date']].head())

print("\n--- Expiry breakdown ---")
for expiry in sorted(merged_df['expiry_date'].unique()):
    cnt = len(merged_df[merged_df['expiry_date'] == expiry])
    print(f"  {pd.to_datetime(expiry).strftime('%Y-%m-%d')}  →  {cnt:>8,} rows")

---
## Cell 10 — Data Integrity Check (No Synthetic Data)
> Verifies that price columns contain real API values and no anomalous fills.

In [ ]:
print("=" * 70)
print("  DATA INTEGRITY CHECK")
print("=" * 70)

price_cols = ['open','high','low','close','volume']
for col in price_cols:
    if col in merged_df.columns:
        nans = merged_df[col].isna().sum()
        zeros = (merged_df[col] == 0).sum()
        negatives = (merged_df[col] < 0).sum()
        print(f"  {col:12s}  NaN={nans:>6,}  zero={zeros:>6,}  negative={negatives:>4,}")

# Check index_close
if 'index_close' in merged_df.columns:
    ic_nan = merged_df['index_close'].isna().sum()
    total  = len(merged_df)
    print(f"\n  index_close NaN: {ic_nan:,} / {total:,}  ({100*ic_nan/total:.1f}%)")
    if ic_nan > 0:
        print("  ⚠ NaN means no SENSEX index candle for that date (holidays/weekends)")
        print("    This is EXPECTED behaviour — no synthetic fill is applied.")
    else:
        print("  ✅ All rows have real index_close values from Groww API.")

# Check CE/PE balance
if 'option_type' in merged_df.columns:
    print(f"\n  Option type counts:")
    print(merged_df['option_type'].value_counts().to_string())

print("\n✅ Integrity check complete. Data sourced 100% from Groww API.")